In [182]:
import numpy as np
import random

# Utilization of hospital beds during epidemics

## Primary task

In [183]:
### 
# Is slow for some reason
####

"""
def arr_per_day(t, lambda_function, id_process): 
    time = 0
    arrivals = [(id_process,(t-1))]
    while True: 
        wait = np.random.exponential(lambda_function(t))
        time += wait 

        if time > 1:
            break
        
        arrivals.append((id_process,time + (t-1)))
    
    return arrivals

def arr_per_year(days):
    all_events = []
    for t in range(1,days+1):
        process_1 = arr_per_day(t, lambda_1, 1)
        process_2 = arr_per_day(t, lambda_2, 2)
        process_3 = arr_per_day(t, lambda_3, 3)

        # Merge daily events 

        all_events += process_1 + process_2 + process_3
    
    # sort by time (assumes time is index 2)
    all_events = sorted(all_events, key=lambda x: x[1])

    return all_events

"""


'\ndef arr_per_day(t, lambda_function, id_process): \n    time = 0\n    arrivals = [(id_process,(t-1))]\n    while True: \n        wait = np.random.exponential(lambda_function(t))\n        time += wait \n\n        if time > 1:\n            break\n        \n        arrivals.append((id_process,time + (t-1)))\n    \n    return arrivals\n\ndef arr_per_year(days):\n    all_events = []\n    for t in range(1,days+1):\n        process_1 = arr_per_day(t, lambda_1, 1)\n        process_2 = arr_per_day(t, lambda_2, 2)\n        process_3 = arr_per_day(t, lambda_3, 3)\n\n        # Merge daily events \n\n        all_events += process_1 + process_2 + process_3\n    \n    # sort by time (assumes time is index 2)\n    all_events = sorted(all_events, key=lambda x: x[1])\n\n    return all_events\n\n'

In [184]:
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    # Input: lami is the arrival rate function for ward i. 
    # Output: A list of tuples where the first entry in the tuple is the patient type and the last entry is the arrival time.
    
    # Initialize
    t =0
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []

    #Iterate over the days
    while t < 365: 
        # Find rates
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        # Simulate arrivals for all three patient types. 
        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1

    # Merge list to create one list of all arrivals in a year
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients

def lambda_1(t): 
    return -(1/3650)*t**2 + (1/10)*t 

def lambda_2(t):
    return 1/5*lambda_1(t)

def lambda_3(t):
    return 6

In [185]:
Patients = arrivals_year(lambda_1, lambda_2,lambda_3)

In [186]:
Patients[-1]

(3, 364.90819796250275)

In [187]:
# System of wards - Do not understand how is beds made in in B an C, how many to begynd 
def system(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
   

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

        LOS_A = []
        LOS_B = []
        LOS_C = []

        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)
    

            else:
                # Reallocate patient
                blocked_A += 1
        
        
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)
                

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    LOS_B.append(LOS)
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))

            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


In [188]:
np.random.seed(42)
bedsA = 45 
bedsB = 15 #shoulden't this be 0? 
bedsC = 15
relocated_A, blocked_B, relocated_C, mean_occupied_A, mean_occupied_B, mean_occupied_C  = system(bedsA,bedsB,bedsC, Patients)

relocated_A, blocked_B, relocated_C, mean_occupied_A, mean_occupied_B, mean_occupied_C 

(611,
 89,
 1486,
 np.float64(0.8516412812054),
 np.float64(0.7405067733557942),
 np.float64(0.9740717344002247))

## Primary performance measures

In [279]:
# Crude monte carlo estimator
def probs(bedsA,bedsB,bedsC,n): #still do not get the input 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lambda_1, lambda_2,lambda_3)
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(a/type_A)
        frac_B.append(b/type_B)
        frac_C.append(c/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)

    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    
    all = A + B + C
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

    

In [304]:
np.random.seed(42)
A = 15
B = 15
C = 45
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution A=15, B=15, C=45
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.7544
  Ward B: 0.2120
  Ward C: 0.2132

Average number of relocated patients:
  Ward A: 1675.22
  Ward B: 94.50
  Ward C: 467.54
  Total : 2237.26

Average bed occupancy:
  Ward A: 93.75%
  Ward B: 74.62%
  Ward C: 93.06%


## Sensitivity analysis

In [281]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
# Monte carlo estimator 
def sum_relocated_MC(bedsA,bedsB,bedsC,n):
    A = []
    B = []
    C = []

    for _ in range(n):
        X = arrivals_year(lambda_1,lambda_2,lambda_3)
        A_blok, B_blok, C_blok, _, _, _ = system(bedsA,bedsB,bedsC, X)
        A.append(A_blok)
        B.append(B_blok)
        C.append(C_blok)
    all = np.array(A) + np.array(B) + np.array(C)

    return np.mean(all)

In [282]:
# System of wards - Do not understand how is beds made in in B an C, how many to begynd 
def system_CV(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0


    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []
    LOS_A = []
    LOS_B = []
    LOS_C = []
    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

    
       
        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)
    

            else:
                # Reallocate patient
                blocked_A += 1
        
        
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)
                

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    LOS_B.append(LOS)
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))

            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)

            else:
                # Reallocate patient
                blocked_C += 1

    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A), np.mean(LOS_B), np.mean(LOS_C)

In [283]:
patients = []
for i in range(100):
    patients.append(arrivals_year(lambda_1,lambda_2,lambda_3))

In [284]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
# Control variate 
LOS_mu_A = 8 
LOS_mu_B = 12
LOS_mu_C = 10 

def sum_relocated_CV(bedsA,bedsB,bedsC, patient_flow):
    A = []
    B = []
    C = []

    LOS_A = []
    LOS_B = []
    LOS_C = []
    n = int(len(patient_flow)/2)
    for X in patient_flow[:n]:
        a, b, c, LOS_a, LOS_b, LOS_c = system_CV(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)

        LOS_A.append(LOS_a)
        LOS_B.append(LOS_b)
        LOS_C.append(LOS_c)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    LOS_A = np.array(LOS_A)
    LOS_B = np.array(LOS_B)
    LOS_C = np.array(LOS_C)
    
    # Find ci 
    c_A = -np.cov(A, LOS_A, ddof=1)[0,1] / np.var(LOS_A, ddof=1)
    c_B = -np.cov(B, LOS_B, ddof=1)[0,1] / np.var(LOS_B, ddof=1)
    c_C = -np.cov(C, LOS_C, ddof=1)[0,1] / np.var(LOS_C, ddof=1)

    # Reinitialiazize 
    A = []
    B = []
    C = []

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for X in patient_flow[n:]:
        a, b, c, LOS_a, LOS_b, LOS_c = system_CV(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)

        LOS_A.append(LOS_a)
        LOS_B.append(LOS_b)
        LOS_C.append(LOS_c)

    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    LOS_A = np.array(LOS_A)
    LOS_B = np.array(LOS_B)
    LOS_C = np.array(LOS_C)

    Y_A = A + c_A*(LOS_A - LOS_mu_A)
    Y_B = B + c_B*(LOS_B - LOS_mu_B)
    Y_C = C + c_C*(LOS_C - LOS_mu_C)

    

    return np.mean(Y_A) + np.mean(Y_B) + np.mean(Y_C)

In [285]:
sum_relocated_CV(17,14,44, patients)

np.float64(2437.1107071865836)

In [286]:
# Different bed distributions 
bed_scenarios = [
    [33, 7, 35],  # 1. Traffic-Proportional Base
    [25, 25, 25], # 2. Equal Split
    [10, 10, 55], # 3. Current Setup (Ward C Favored)
    [50, 5, 20],  # 4. Aggressive Ward A Focus
    [65, 5, 5],    # 5. Minimum-Bed Stress Test
    [45, 0, 30],   # 6. Delete ward B
]
n = 100
for bedA, bedB, bedC in bed_scenarios:
    mean_block = sum_relocated_MC(bedA,bedB,bedC,n)
    print(f"Monte Carlo estimator, n = {n}")
    print(f"Mean of blocked patientens(relocated): {mean_block}")
    print(f"Bed distribution A: {bedA}, B: {bedB}, C: {bedC}")
    print()

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2272.61
Bed distribution A: 33, B: 7, C: 35

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2339.46
Bed distribution A: 25, B: 25, C: 25

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2394.34
Bed distribution A: 10, B: 10, C: 55

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2390.17
Bed distribution A: 50, B: 5, C: 20

Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2621.79
Bed distribution A: 65, B: 5, C: 5



/var/folders/x2/5q06y6lx7pl1m_kc24264lbh0000gn/T/ipykernel_81370/193675139.py:100: RuntimeWarning: invalid value encountered in scalar divide
  return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


Monte Carlo estimator, n = 100
Mean of blocked patientens(relocated): 2463.0
Bed distribution A: 45, B: 0, C: 30



In [287]:
# Different bed distributions 
bed_scenarios = [
    [33, 7, 35],  # 1. Traffic-Proportional Base
    [25, 25, 25], # 2. Equal Split
    [10, 10, 55], # 3. Current Setup (Ward C Favored)
    [50, 5, 20],  # 4. Aggressive Ward A Focus
    [65, 5, 5],    # 5. Minimum-Bed Stress Test
    [45, 0, 30],   # 6. Delete ward B
]
n = len(patients)
for bedA, bedB, bedC in bed_scenarios:
    mean_block = sum_relocated_CV(bedA,bedB,bedC, patients)
    print(f"Control variate estimator, n = {n}")
    print(f"Mean of blocked patientens(relocated): {mean_block}")
    print(f"Bed distribution A: {bedA}, B: {bedB}, C: {bedC}")
    print()

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2518.8720156767004
Bed distribution A: 33, B: 7, C: 35

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2602.6632439793184
Bed distribution A: 25, B: 25, C: 25

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2659.5347081225445
Bed distribution A: 10, B: 10, C: 55

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2653.612482075743
Bed distribution A: 50, B: 5, C: 20

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2789.9158342129954
Bed distribution A: 65, B: 5, C: 5

Control variate estimator, n = 100
Mean of blocked patientens(relocated): 2678.146143271608
Bed distribution A: 45, B: 0, C: 30



### Task 1: Find optimal bed distribution
Based on sum of relocated patients 

In [ ]:
# Minimize the sum of relocated patients for different bed distribution

def generate_start_bed_dist(bed_cap, patients, n_bed_dist, n_dist_prop): # do course grid instead
    best_bed= []

    for run in range(n_bed_dist):

        best_val = float("inf")
        best_alloc = None

        for _ in range(n_dist_prop):
            A = random.randint(1, bed_cap - 2)
            B = random.randint(1, bed_cap - A)
            C = bed_cap - A - B

            if C <=0: 
                continue

            val = sum_relocated_CV(A, B, C, patients) 

            if val < best_val:
                best_val = val
                best_alloc = (A, B, C)
            
        print(f"Iteration {run}\{n_bed_dist - 1}")
        best_bed.append((best_alloc, best_val))
    return best_bed

In [294]:
n_bed_dist = 10
n_dist_prop = 100
patients_10 = [] #maybe do with 100
for i in range(10):
    patients_10.append(arrivals_year(lambda_1,lambda_2,lambda_3))

bed_cap = 75


In [295]:
best_bed = generate_start_bed_dist(bed_cap, patients_10, n_bed_dist, n_dist_prop)


Iteration 0\9
Iteration 1\9
Iteration 2\9
Iteration 3\9
Iteration 4\9
Iteration 5\9
Iteration 6\9
Iteration 7\9
Iteration 8\9
Iteration 9\9


In [296]:
best_bed

[((51, 20, 4), np.float64(1899.5590276589364)),
 ((64, 9, 2), np.float64(2006.183596556959)),
 ((36, 12, 27), np.float64(1921.5145458877432)),
 ((31, 27, 17), np.float64(1920.4166029606888)),
 ((45, 26, 4), np.float64(2144.886771288015)),
 ((57, 17, 1), np.float64(2013.1121847494946)),
 ((33, 15, 27), np.float64(1096.3308911778856)),
 ((48, 1, 26), np.float64(2100.384625313878)),
 ((39, 15, 21), np.float64(2033.3422514671659)),
 ((59, 1, 15), np.float64(1584.361605840864))]

In [297]:
def optimize_beds_stepwise(patient_data, start_dist):
    current_dist = list(start_dist)
    best_dist = list(start_dist)
    
    # Get baseline metrics using your updated function
    best_score = sum_relocated_CV(current_dist[0],current_dist[1], current_dist[2],patient_data)
    print(f"Starting Baseline {current_dist}: {best_score} total issues")

    improved = True
    while improved:
        improved = False
        neighbors = []
        for i in range(3):
            for j in range(3):
                # Ensure we don't drop a ward's bed count below 0
                if i != j and current_dist[i] > 0: 
                    test_dist = list(current_dist)
                    test_dist[i] -= 1
                    test_dist[j] += 1
                    neighbors.append(test_dist)
        
        for neighbor in neighbors:
            print(f"Testing adjustment: {neighbor}...")
            score = sum_relocated_CV(neighbor[0],neighbor[1], neighbor[2],patient_data)
            
            if score < best_score:
                best_score = score
                best_dist = neighbor
                improved = True
        
        if improved:
            current_dist = list(best_dist)
            print(f"Found better distribution: {current_dist} with {best_score} issues")
            
    return best_dist, best_score


In [ ]:
# simulate patients 
patients_100 = []
for i in range(100):
    patients_100.append(arrivals_year(lambda_1,lambda_2,lambda_3))

In [ ]:
# Search in neighboorhood 
after_search_best = []
for start_dist, _ in best_bed:
    opt_dist, opt_score = optimize_beds_stepwise(patients_100,  start_dist)
    after_search_best.append((opt_dist, opt_score))
    print(f"Start distribution {start_dist}")
    print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")

Starting Baseline [51, 20, 4]: 2661.7321534737803 total issues
Testing adjustment: [50, 21, 4]...
Testing adjustment: [50, 20, 5]...
Testing adjustment: [52, 19, 4]...
Testing adjustment: [51, 19, 5]...
Testing adjustment: [52, 20, 3]...
Testing adjustment: [51, 21, 3]...
Found better distribution: [51, 19, 5] with 2528.9631579179695 issues
Testing adjustment: [50, 20, 5]...
Testing adjustment: [50, 19, 6]...
Testing adjustment: [52, 18, 5]...
Testing adjustment: [51, 18, 6]...
Testing adjustment: [52, 19, 4]...
Testing adjustment: [51, 20, 4]...
Start distribution (51, 20, 4)

Final Optimal Distribution: [51, 19, 5] with 2528.9631579179695 total problems
Starting Baseline [64, 9, 2]: 2739.6888823160925 total issues
Testing adjustment: [63, 10, 2]...
Testing adjustment: [63, 9, 3]...
Testing adjustment: [65, 8, 2]...
Testing adjustment: [64, 8, 3]...
Testing adjustment: [65, 9, 1]...
Testing adjustment: [64, 10, 1]...
Found better distribution: [64, 8, 3] with 2628.682368100228 issues


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Testing adjustment: [57, 18, 0]...
Found better distribution: [56, 17, 2] with 2538.687375053513 issues
Testing adjustment: [55, 18, 2]...
Testing adjustment: [55, 17, 3]...
Testing adjustment: [57, 16, 2]...
Testing adjustment: [56, 16, 3]...
Testing adjustment: [57, 17, 1]...
Testing adjustment: [56, 18, 1]...
Start distribution (57, 17, 1)

Final Optimal Distribution: [56, 17, 2] with 2538.687375053513 total problems
Starting Baseline [33, 15, 27]: 2416.0783112940126 total issues
Testing adjustment: [32, 16, 27]...
Testing adjustment: [32, 15, 28]...
Testing adjustment: [34, 14, 27]...
Testing adjustment: [33, 14, 28]...
Testing adjustment: [34, 15, 26]...
Testing adjustment: [33, 16, 26]...
Start distribution (33, 15, 27)

Final Optimal Distribution: [33, 15, 27] with 2416.0783112940126 total problems
Starting Baseline [48, 1, 26]: 2789.073655410157 total issues
Testing adjustment: [47, 2, 26]...
Testing adjustment: [47, 1, 27]...
Testing adjustment: [49, 0, 26]...
Testing adjustme

In [300]:
best_sorted = sorted(after_search_best, key=lambda x: x[1])

In [302]:
best_sorted

[([40, 14, 21], np.float64(2340.200425954783)),
 ([36, 13, 26], np.float64(2354.823940406631)),
 ([33, 15, 27], np.float64(2416.0783112940126)),
 ([33, 26, 16], np.float64(2507.3786901829676)),
 ([47, 2, 26], np.float64(2528.5056263946617)),
 ([51, 19, 5], np.float64(2528.9631579179695)),
 ([56, 17, 2], np.float64(2538.687375053513)),
 ([63, 8, 4], np.float64(2602.399670289282)),
 ([46, 25, 4], np.float64(2602.518907911991)),
 ([59, 1, 15], np.float64(2646.7124380843306))]

In [ ]:
# Performance measure of old found best bed distribution for cap = 75 
np.random.seed(42)
A = best_sorted[0][0][0]
B = best_sorted[0][0][1]
C = best_sorted[0][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution A=40, B=14, C=21
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.3493
  Ward B: 0.2473
  Ward C: 0.6162

Average number of relocated patients:
  Ward A: 775.88
  Ward B: 110.67
  Ward C: 1350.87
  Total : 2237.43

Average bed occupancy:
  Ward A: 85.54%
  Ward B: 76.05%
  Ward C: 97.01%


In [ ]:
# Exponential length of stay distributial 

def system_exponential(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
    

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.exponential(scale=8)
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=12)
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=10)
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC

# Crude monte carlo estimator
def probs_expontial(bedsA,bedsB,bedsC,n): #still do not get the input 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lambda_1, lambda_2,lambda_3)
        A, B, C, bed_frac_A,bed_frac_B,bed_frac_C = system_exponential(bedsA,bedsB,bedsC, X)
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(A/type_A)
        frac_B.append(B/type_B)
        frac_C.append(C/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

In [ ]:
# Exponential with old found bed distributions
A = best_sorted[0][0][0]
B = best_sorted[0][0][1]
C = best_sorted[0][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs_expontial(A,B,C,1000)
print(f"Results for bed distribution exponential  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution exponential  A=40, B=14, C=21
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.4123
  Ward B: 0.2934
  Ward C: 0.6504

Average number of relocated patients:
  Ward A: 941.00
  Ward B: 145.00
  Ward C: 1379.00
  Total : 2465.00

Average bed occupancy:
  Ward A: 87.37%
  Ward B: 78.61%
  Ward C: 97.36%


In [ ]:
# Different bed capacity 

bed_cap = [50, 75, 100]

# Simulate patients 
n_bed_dist = 10 
n_dist_prop = 100
patients_10 = [] #maybe do with 100
for i in range(10):
    patients_10.append(arrivals_year(lambda_1,lambda_2,lambda_3))

patients_100 = []
for i in range(100):
    patients_100.append(arrivals_year(lambda_1,lambda_2,lambda_3))

    


In [ ]:
# Find proposed bed distribution
start_dist = []
for cap in bed_cap:
    start_dist.append(generate_start_bed_dist(cap, patients_10, n_bed_dist, n_dist_prop))
    print(f"done capacity {cap}")



Iteration 0\9
Iteration 1\9
Iteration 2\9
Iteration 3\9
Iteration 4\9
Iteration 5\9
Iteration 6\9
Iteration 7\9
Iteration 8\9
Iteration 9\9
done capacity 50
Iteration 0\9
Iteration 1\9
Iteration 2\9
Iteration 3\9
Iteration 4\9
Iteration 5\9
Iteration 6\9
Iteration 7\9
Iteration 8\9
Iteration 9\9
done capacity 75
Iteration 0\9
Iteration 1\9
Iteration 2\9
Iteration 3\9
Iteration 4\9
Iteration 5\9
Iteration 6\9
Iteration 7\9
Iteration 8\9
Iteration 9\9
done capacity 100


In [ ]:
# Do search in neighboorhood of the proposed bed distributions 
best_overall = []
for dist in start_dist: 
    after_search_best = []
    for start_dist_i, _ in dist:
        opt_dist, opt_score = optimize_beds_stepwise(patients_100,  start_dist_i)
        after_search_best.append((opt_dist, opt_score))
        print(f"Start distribution {start_dist_i}")
        print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")
    
    best_overall.append(sorted(after_search_best, key=lambda x: x[1])[0])

Starting Baseline [15, 7, 28]: 3361.4164884672427 total issues
Testing adjustment: [14, 8, 28]...
Testing adjustment: [14, 7, 29]...
Testing adjustment: [16, 6, 28]...
Testing adjustment: [15, 6, 29]...
Testing adjustment: [16, 7, 27]...
Testing adjustment: [15, 8, 27]...
Found better distribution: [14, 7, 29] with 3342.2028770718907 issues
Testing adjustment: [13, 8, 29]...
Testing adjustment: [13, 7, 30]...
Testing adjustment: [15, 6, 29]...
Testing adjustment: [14, 6, 30]...
Testing adjustment: [15, 7, 28]...
Testing adjustment: [14, 8, 28]...
Found better distribution: [14, 8, 28] with 3341.829455985059 issues
Testing adjustment: [13, 9, 28]...
Testing adjustment: [13, 8, 29]...
Testing adjustment: [15, 7, 28]...
Testing adjustment: [14, 7, 29]...
Testing adjustment: [15, 8, 27]...
Testing adjustment: [14, 9, 27]...
Found better distribution: [13, 8, 29] with 3334.913312665321 issues
Testing adjustment: [12, 9, 29]...
Testing adjustment: [12, 8, 30]...
Testing adjustment: [14, 7, 2

In [312]:
best_overall

[([28, 15, 7], np.float64(3202.4860148442754)),
 ([21, 16, 38], np.float64(2350.3150486954764)),
 ([41, 22, 37], np.float64(1492.4281708665885))]

In [ ]:
# 50 beds performance measures 
A = best_overall[0][0][0]
B = best_overall[0][0][1]
C = best_overall[0][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 50 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 50 beds  A=28, B=15, C=7
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.5741
  Ward B: 0.2570
  Ward C: 0.8824

Average number of relocated patients:
  Ward A: 1346.00
  Ward B: 93.00
  Ward C: 1880.00
  Total : 3319.00

Average bed occupancy:
  Ward A: 90.95%
  Ward B: 77.34%
  Ward C: 98.11%


In [ ]:
# 75 bed performance measures 
A = best_overall[1][0][0]
B = best_overall[1][0][1]
C = best_overall[1][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 75 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 75 beds  A=21, B=16, C=38
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.6766
  Ward B: 0.2238
  Ward C: 0.3804

Average number of relocated patients:
  Ward A: 1574.00
  Ward B: 105.00
  Ward C: 943.00
  Total : 2622.00

Average bed occupancy:
  Ward A: 92.80%
  Ward B: 75.92%
  Ward C: 95.71%


In [ ]:
# 100 beds performance measures 

A = best_overall[2][0][0]
B = best_overall[2][0][1]
C = best_overall[2][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 100 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 100 beds  A=41, B=22, C=37
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.3408
  Ward B: 0.0780
  Ward C: 0.3961

Average number of relocated patients:
  Ward A: 738.00
  Ward B: 24.00
  Ward C: 902.00
  Total : 1664.00

Average bed occupancy:
  Ward A: 86.51%
  Ward B: 66.70%
  Ward C: 95.86%


In [ ]:
# Exponential distribution 75 beds 
A = best_overall[1][0][0]
B = best_overall[1][0][1]
C = best_overall[1][0][2]

fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs_expontial(A,B,C,1000)
print(f"Results for bed distribution exponential  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

### Sensitivity analysis - results (in my opnionen)

In [ ]:
# Different bed capacity but with grid search 

In [327]:
def generate_grid_starts(bed_cap, patients, step=5, min_beds=1):
    """
    Generate evenly spaced feasible bed distributions.

    Parameters
    ----------
    bed_cap : int
        Total number of beds.
    step : int
        Grid spacing.
    min_beds : int
        Minimum beds allowed in each ward.

    Returns
    -------
    list of tuples
        [(A,B,C), ...]
    """

    starts = []

    for A in range(min_beds, bed_cap - 2*min_beds + 1, step):
        for B in range(min_beds, bed_cap - A - min_beds + 1, step):
            C = bed_cap - A - B

            if C >= min_beds:
                starts.append((A, B, C))
    
    results = []

    i = 0 
    for A, B, C in starts:
        val = sum_relocated_CV(A, B, C, patients)
        results.append(((A, B, C), val))
        print(f"Iteration {i}\{len(starts)}")
        i += 1 
    
    results.sort(key=lambda x: x[1])

    return results[:10]

In [329]:

bed_cap = [50, 75, 100]

patients_100 = []
for i in range(100):
    patients_100.append(arrivals_year(lambda_1,lambda_2,lambda_3))

best_start_grid = []
for cap in bed_cap:
    best_start_grid.append(generate_grid_starts(cap, patients_100, step=5, min_beds=1))
    print(f"done capacity {cap}")

best_start_grid

Iteration 0\55
Iteration 1\55
Iteration 2\55
Iteration 3\55
Iteration 4\55
Iteration 5\55
Iteration 6\55
Iteration 7\55
Iteration 8\55
Iteration 9\55
Iteration 10\55
Iteration 11\55
Iteration 12\55
Iteration 13\55
Iteration 14\55
Iteration 15\55
Iteration 16\55
Iteration 17\55
Iteration 18\55
Iteration 19\55
Iteration 20\55
Iteration 21\55
Iteration 22\55
Iteration 23\55
Iteration 24\55
Iteration 25\55
Iteration 26\55
Iteration 27\55
Iteration 28\55
Iteration 29\55
Iteration 30\55
Iteration 31\55
Iteration 32\55
Iteration 33\55
Iteration 34\55
Iteration 35\55
Iteration 36\55
Iteration 37\55
Iteration 38\55
Iteration 39\55
Iteration 40\55
Iteration 41\55
Iteration 42\55
Iteration 43\55
Iteration 44\55
Iteration 45\55
Iteration 46\55
Iteration 47\55
Iteration 48\55
Iteration 49\55
Iteration 50\55
Iteration 51\55
Iteration 52\55
Iteration 53\55
Iteration 54\55
done capacity 50
Iteration 0\120
Iteration 1\120
Iteration 2\120
Iteration 3\120
Iteration 4\120
Iteration 5\120
Iteration 6\120
I

[[((21, 16, 13), np.float64(3214.9032670577776)),
  ((26, 16, 8), np.float64(3246.593007255834)),
  ((16, 16, 18), np.float64(3256.2258142452206)),
  ((36, 11, 3), np.float64(3258.6625992035215)),
  ((31, 6, 13), np.float64(3268.5703571055637)),
  ((21, 11, 18), np.float64(3275.7014163450813)),
  ((31, 11, 8), np.float64(3277.1564736251103)),
  ((16, 11, 23), np.float64(3286.924425779525)),
  ((31, 16, 3), np.float64(3296.2041901858956)),
  ((26, 11, 13), np.float64(3306.824681709337))],
 [((41, 16, 18), np.float64(2279.1854865750865)),
  ((26, 16, 33), np.float64(2320.2534484593293)),
  ((21, 21, 33), np.float64(2356.009558079936)),
  ((31, 11, 33), np.float64(2362.0720480214486)),
  ((26, 21, 28), np.float64(2391.5702555419257)),
  ((21, 16, 38), np.float64(2395.484824624168)),
  ((16, 11, 48), np.float64(2408.392129700649)),
  ((36, 16, 23), np.float64(2409.8862882084477)),
  ((46, 11, 18), np.float64(2439.7147237453455)),
  ((51, 16, 8), np.float64(2441.750853582767))],
 [((31, 21,

In [336]:
best_start_grid_save = best_start_grid.copy()

In [338]:
# Generate new patients to simulate 

patients_100 = []
for i in range(100):
    patients_100.append(arrivals_year(lambda_1,lambda_2,lambda_3))

best_overall_grid = []
for dist in best_start_grid: 
    after_search_best_grid = []
    for start_dist_i, _ in dist:
        opt_dist, opt_score = optimize_beds_stepwise(patients_100,  start_dist_i)
        after_search_best_grid.append((opt_dist, opt_score))
        print(f"Start distribution {start_dist_i}")
        print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")
    
    best_overall_grid.append(sorted(after_search_best_grid, key=lambda x: x[1])[0])

Starting Baseline [21, 16, 13]: 3305.6539072788883 total issues
Testing adjustment: [20, 17, 13]...
Testing adjustment: [20, 16, 14]...
Testing adjustment: [22, 15, 13]...
Testing adjustment: [21, 15, 14]...
Testing adjustment: [22, 16, 12]...
Testing adjustment: [21, 17, 12]...
Found better distribution: [22, 15, 13] with 3191.657484405103 issues
Testing adjustment: [21, 16, 13]...
Testing adjustment: [21, 15, 14]...
Testing adjustment: [23, 14, 13]...
Testing adjustment: [22, 14, 14]...
Testing adjustment: [23, 15, 12]...
Testing adjustment: [22, 16, 12]...
Start distribution (21, 16, 13)

Final Optimal Distribution: [22, 15, 13] with 3191.657484405103 total problems
Starting Baseline [26, 16, 8]: 3346.9052967190446 total issues
Testing adjustment: [25, 17, 8]...
Testing adjustment: [25, 16, 9]...
Testing adjustment: [27, 15, 8]...
Testing adjustment: [26, 15, 9]...
Testing adjustment: [27, 16, 7]...
Testing adjustment: [26, 17, 7]...
Found better distribution: [27, 16, 7] with 3278.

In [ ]:
best_overall_grid_save = best_overall_grid.copy()


[([22, 15, 13], np.float64(3191.657484405103)),
 ([32, 10, 33], np.float64(2329.949522591923)),
 ([41, 20, 39], np.float64(1507.119439548624))]

In [342]:
# 50 beds performance measures 
A = best_overall_grid[0][0][0]
B = best_overall_grid[0][0][1]
C = best_overall_grid[0][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 50 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

# 75 beds performance measures 
A = best_overall_grid[1][0][0]
B = best_overall_grid[1][0][1]
C = best_overall_grid[1][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 75 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

# 100 beds performance measures 
A = best_overall_grid[2][0][0]
B = best_overall_grid[2][0][1]
C = best_overall_grid[2][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 100 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

# 75 beds exponential distribution performance measure 

A = best_overall_grid[1][0][0]
B = best_overall_grid[1][0][1]
C = best_overall_grid[1][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs_expontial(A,B,C,1000)
print(f"Results for bed distribution capacity 75 beds exponential  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 50 beds  A=22, B=15, C=13
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.6248
  Ward B: 0.2097
  Ward C: 0.7605

Average number of relocated patients:
  Ward A: 1387.54
  Ward B: 93.52
  Ward C: 1666.21
  Total : 3147.26

Average bed occupancy:
  Ward A: 91.77%
  Ward B: 74.53%
  Ward C: 97.56%
Results for bed distribution capacity 75 beds  A=32, B=10, C=33
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.5137
  Ward B: 0.4117
  Ward C: 0.4045

Average number of relocated patients:
  Ward A: 1139.65
  Ward B: 183.14
  Ward C: 885.65
  Total : 2208.45

Average bed occupancy:
  Ward A: 88.99%
  Ward B: 81.68%
  Ward C: 95.65%
Results for bed distribution capacity 100 beds  A=41, B=20, C=39
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.2874
  Ward B: 0.0808
  Ward C: 0.3053

Average number of relocated patients:
  Ward A: 638.64